In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
print(gold_schema,silver_schema,bronze_schema)

In [0]:
dbutils.widgets.text('catalog','agmarknet')
dbutils.widgets.text('data source','daily_prices')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data source')

print(catalog)
print(data_source)

In [0]:
# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.fact_{data_source}"

bronze_table, silver_table, gold_table

In [0]:
base_path = f's3://agmarknet-pc/AgmarkIncremental'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

###Bronze

Read daily file from S3 landing folder

In [0]:
df1 = (spark.read
       .option("header", "true")
       .option("inferSchema", "false")
       .option("delta.enableChangeDataFeed", "true")
       .option("delimiter", ",")
       .option("multiline", "true")
        .csv(landing_path)
        .withColumn('read_timestamp',F.current_timestamp())
        .select("*", "_metadata.file_name")
)

In [0]:
display(df1)

In [0]:
df1.printSchema()

In [0]:
### Save the df1 to bronze table
df1.write \
.format("delta") \
.option("delta.enableChangeDataFeed", "true") \
.mode("append") \
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

Staging table to process just the arrived incremenal data

In [0]:
df1.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeschema","true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

###Silver

In [0]:
df_price = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")
display(df_price)

Silver Layer Transformations

In [0]:
# cast min price , max price, modal price to double
#cast commodity code to int

df_price = df_price.withColumn("Min_Price", F.col("Min_Price").cast("double")) \
    .withColumn("Max_Price",F.col("Max_Price").cast("double")) \
    .withColumn("Modal_Price",F.col("Modal_Price").cast("double")) \
    .withColumn("Commodity_Code",F.col("Commodity_Code").cast("int"))

In [0]:
df_price.printSchema()

In [0]:
# analyze date formats before transformation
df_silver_dates = spark.sql(f'select * from {catalog}.{bronze_schema}.staging_{data_source}')

df_formats = (
    df_silver_dates.withColumn(
        "date_format",
        F.when(F.col("Arrival_Date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
         .when(F.col("Arrival_Date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
         .when(F.col("Arrival_Date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
         .when(F.col("Arrival_Date").rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
         .when(F.col("Arrival_Date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
         .otherwise("Unknown")
    )
)

df_formats.groupBy("date_format").count().show(truncate=False)

In [0]:
df_price = df_price.withColumn("Arrival_Date",F.to_date("Arrival_Date",'dd/MM/yyyy'))

### #Save/Append the latest data to silver table

In [0]:
silver_delta = DeltaTable.forName(spark,silver_table)
silver_delta.alias("silver").merge(df_price.alias("new"),
                                   """
                                   silver.Arrival_Date= new.Arrival_Date AND
                                   silver.Commodity_Code =new.Commodity_Code
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Staging silver table to process just the arrived incremenal data

In [0]:
# stagging for incremental data

df_price.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

###Gold

In [0]:
silverdf = spark.sql(f"select * from {catalog}.{silver_schema}.staging_{data_source}")
marketdf = spark.sql(f"select * from {catalog}.{gold_schema}.dim_market")

In [0]:
fact_df = (
    silverdf.alias("f")
    .join(
        marketdf.alias("d"),
        on=["Market", "District", "State"],
        how = "left"
    )
    .select("Arrival_Date"
            ,"Commodity_Code"
            ,"d.Market_Code"
            ,"Min_Price"
            ,"Max_Price"
            ,"Modal_Price"
            )

)

In [0]:
fact_df.show(10)

In [0]:
fact_df.count()

###writing/append new data to gold table

In [0]:
gold_delta = DeltaTable.forName(spark,gold_table)
gold_delta.alias("gold").merge(fact_df.alias("new"),
                                   """
                                   gold.Arrival_Date= new.Arrival_Date AND
                                   gold.Commodity_Code =new.Commodity_Code
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
golddf = spark.sql(f"select * from {catalog}.{gold_schema}.fact_{data_source}")
golddf.count()

## Cleanup

In [0]:
%sql
DROP TABLE agmarknet.bronze.staging_daily_prices;

In [0]:
%sql
drop table agmarknet.silver.staging_daily_prices